# MLflow Quickstart

Train a scikit-learn model, then use MLflow to track params/metrics, log the model, register it and load it back for predictions.

Start the tracking server first (from the repo root, in a terminal):

```
mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```

Open http://127.0.0.1:5000 to see the UI.

In [1]:
import mlflow
import pandas as pd
from mlflow.models import infer_signature

from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

## 1. Connect to the tracking server

In [3]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("MLflow Quickstart")
print("MLflow Version:", mlflow.__version__)
print("MLflow Tracking URI:", mlflow.get_tracking_uri())

MLflow Version: 3.16.1
MLflow Tracking URI: http://127.0.0.1:5000


## 2. Train a model

In [4]:
X, y = datasets.load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "random_state": 8888,
}

lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")
print("Accuracy:", accuracy)
print("F1 (macro):", f1)

Accuracy: 1.0
F1 (macro): 1.0


## 3. Log the run: params, metrics and the model

In [5]:
with mlflow.start_run(run_name="logreg-baseline") as run:
    mlflow.log_params(params)
    mlflow.log_metrics({"accuracy": accuracy, "f1_macro": f1})
    mlflow.set_tag("Training Info", "Basic LR model for iris data")

    signature = infer_signature(X_train, lr.predict(X_train))

    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        name="iris_model",
        signature=signature,
        input_example=X_train.head(),
        registered_model_name="tracking-quickstart",
    )

print("Run ID:", run.info.run_id)
print("Model URI:", model_info.model_uri)

Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2026/09/27 16:47:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 2


🏃 View run logreg-baseline at: http://127.0.0.1:5000/#/experiments/1/runs/95a58b521b0f410fb129493547347952
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 95a58b521b0f410fb129493547347952
Model URI: models:/m-e64e4a9881914c22b24a0430197ce3c6


Created version '2' of model 'tracking-quickstart'.


## 4. Load the model back and predict

In [6]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
predictions = loaded_model.predict(X_test)

iris_feature_names = datasets.load_iris().feature_names
result = pd.DataFrame(X_test, columns=iris_feature_names)
result["actual_class"] = y_test
result["predicted_class"] = predictions
result.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual_class,predicted_class
73,6.1,2.8,4.7,1.2,1,1
18,5.7,3.8,1.7,0.3,0,0
118,7.7,2.6,6.9,2.3,2,2
78,6.0,2.9,4.5,1.5,1,1
76,6.8,2.8,4.8,1.4,1,1


## 5. Autologging (optional)

`autolog()` records params, metrics and the model automatically, with no manual `log_*` calls.

In [7]:
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="logreg-autolog"):
    lr_auto = LogisticRegression(solver="lbfgs", max_iter=500, C=0.5, random_state=8888)
    lr_auto.fit(X_train, y_train)
    lr_auto.score(X_test, y_test)

mlflow.sklearn.autolog(disable=True)

2026/09/27 16:48:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
c:\Users\laksh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:3424: FutureWarning: `y_pred` was renamed to `y_proba` in version 1.9 and will be removed in 1.11. Use `y_proba` instead.
  warnings.warn(


🏃 View run logreg-autolog at: http://127.0.0.1:5000/#/experiments/1/runs/0cd240a3c0d64a3eb60957ed12aee5d6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## 6. Compare runs

In [8]:
runs = mlflow.search_runs(experiment_names=["MLflow Quickstart"])
runs[["run_id", "tags.mlflow.runName", "metrics.accuracy", "metrics.training_accuracy_score", "start_time"]]

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.training_accuracy_score,start_time
0,0cd240a3c0d64a3eb60957ed12aee5d6,logreg-autolog,NaN,0.966667,2026-09-27 11:18:02.814000+00:00
1,95a58b521b0f410fb129493547347952,logreg-baseline,1.0,NaN,2026-09-27 11:17:28.043000+00:00
2,cf26d1c00f2a40d4bb91b67d9d524164,logreg-baseline,1.0,NaN,2026-09-27 11:14:57.642000+00:00
